In [6]:
!pip install -q ultralytics opencv-python matplotlib pyyaml tqdm

In [7]:
from pathlib import Path
import json
import shutil
import random

import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
import yaml

from ultralytics import YOLO

### paths ###
# change this to colab's project root
ROOT = Path("/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement")
print("Project root:", ROOT)

# where RUOD was extracted
RUOD_ROOT = ROOT / "data" / "RUOD"
print("RUOD root:", RUOD_ROOT)

# RUOD image & annotation paths
RUOD_IMG_TRAIN = RUOD_ROOT / "RUOD_pic" / "train"
RUOD_IMG_TEST  = RUOD_ROOT / "RUOD_pic" / "test"

RUOD_ANN_TRAIN = RUOD_ROOT / "RUOD_ANN" / "instances_train.json"
RUOD_ANN_TEST  = RUOD_ROOT / "RUOD_ANN" / "instances_test.json"

# check paths
for p in [RUOD_IMG_TRAIN, RUOD_IMG_TEST, RUOD_ANN_TRAIN, RUOD_ANN_TEST]:
    print(p, "exists:", p.exists())

Project root: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement
RUOD root: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD
/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_pic/train exists: True
/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_pic/test exists: True
/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_ANN/instances_train.json exists: True
/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_ANN/instances_test.json exists: True


In [8]:
### load COCO-style json and inspect categories ###

def load_coco_json(path):
    with open(path, "r") as f:
        data = json.load(f)
    return data

coco_train = load_coco_json(RUOD_ANN_TRAIN)
coco_test  = load_coco_json(RUOD_ANN_TEST)

print("Keys in COCO train json:", coco_train.keys())
print("Num train images:", len(coco_train["images"]))
print("Num train annotations:", len(coco_train["annotations"]))
print("Num categories:", len(coco_train["categories"]))

print("\nCategories:")
for cat in coco_train["categories"]:
    print(f"  id={cat['id']}, name={cat['name']}")

Keys in COCO train json: dict_keys(['images', 'type', 'annotations', 'categories'])
Num train images: 9800
Num train annotations: 51935
Num categories: 10

Categories:
  id=1, name=holothurian
  id=2, name=echinus
  id=3, name=scallop
  id=4, name=starfish
  id=5, name=fish
  id=6, name=corals
  id=7, name=diver
  id=8, name=cuttlefish
  id=9, name=turtle
  id=10, name=jellyfish


In [9]:
### category mapping (COCO id -> YOLO index) ###
coco_categories = coco_train["categories"]

# YOLO classes: 0..(N-1)
coco_id_to_yolo_id = {}
yolo_id_to_name = {}

for i, cat in enumerate(sorted(coco_categories, key=lambda c: c["id"])):
    coco_id = cat["id"]
    coco_id_to_yolo_id[coco_id] = i
    yolo_id_to_name[i] = cat["name"]

print("COCO -> YOLO id:")
for cid, yid in coco_id_to_yolo_id.items():
    print(f"  COCO id {cid} -> YOLO id {yid} ({yolo_id_to_name[yid]})")

COCO -> YOLO id:
  COCO id 1 -> YOLO id 0 (holothurian)
  COCO id 2 -> YOLO id 1 (echinus)
  COCO id 3 -> YOLO id 2 (scallop)
  COCO id 4 -> YOLO id 3 (starfish)
  COCO id 5 -> YOLO id 4 (fish)
  COCO id 6 -> YOLO id 5 (corals)
  COCO id 7 -> YOLO id 6 (diver)
  COCO id 8 -> YOLO id 7 (cuttlefish)
  COCO id 9 -> YOLO id 8 (turtle)
  COCO id 10 -> YOLO id 9 (jellyfish)


In [10]:
### make YOLO dirs ###

YOLO_ROOT   = ROOT / "data" / "ruod_yolo"
YOLO_IMG_TR = YOLO_ROOT / "images" / "train"
YOLO_IMG_VAL= YOLO_ROOT / "images" / "val"
YOLO_LAB_TR = YOLO_ROOT / "labels" / "train"
YOLO_LAB_VAL= YOLO_ROOT / "labels" / "val"

for d in [YOLO_IMG_TR, YOLO_IMG_VAL, YOLO_LAB_TR, YOLO_LAB_VAL]:
    d.mkdir(parents=True, exist_ok=True)
    print("Ensured directory:", d)

Ensured directory: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/ruod_yolo/images/train
Ensured directory: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/ruod_yolo/images/val
Ensured directory: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/ruod_yolo/labels/train
Ensured directory: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/ruod_yolo/labels/val


In [11]:
### functions to convert COCO -> YOLO ###

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """
    Convert COCO bbox [x_min, y_min, width, height] to:
    YOLO [x_center/img_w, y_center/img_h, w/img_w, h/img_h]
    """
    x, y, w, h = bbox
    x_c = x + w / 2.0
    y_c = y + h / 2.0

    return [
        x_c / img_w,
        y_c / img_h,
        w   / img_w,
        h   / img_h,
    ]


def build_image_dict(coco):
    """
    Build lookup dicts:
    - id_to_image: image_id -> image_info (width, height, file_name)
    - img_id_to_anns: image_id -> list of annotation dicts
    """
    id_to_image = {img["id"]: img for img in coco["images"]}

    img_id_to_anns = {}
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        if img_id not in img_id_to_anns:
            img_id_to_anns[img_id] = []
        img_id_to_anns[img_id].append(ann)

    return id_to_image, img_id_to_anns

In [12]:
### main COCO -> YOLO conversion for one split ###

def convert_coco_split_to_yolo(
    coco_json_path,
    img_src_dir,
    out_img_dir,
    out_label_dir,
    coco_id_to_yolo_id,
):
    coco = load_coco_json(coco_json_path)
    id_to_image, img_id_to_anns = build_image_dict(coco)

    print(f"Converting {len(id_to_image)} images from", img_src_dir)
    n_no_ann = 0

    for img_id, img_info in tqdm(id_to_image.items()):
        file_name = img_info["file_name"]
        img_w = img_info["width"]
        img_h = img_info["height"]

        src_path = img_src_dir / file_name
        dst_img_path = out_img_dir / file_name

        if not src_path.exists():
            print("[WARN] missing image:", src_path)
            continue

        # copy image
        if not dst_img_path.exists():
            shutil.copy2(src_path, dst_img_path)

        anns = img_id_to_anns.get(img_id, [])
        label_lines = []

        for ann in anns:
            if ann.get("iscrowd", 0) == 1:
                continue

            coco_cat_id = ann["category_id"]
            if coco_cat_id not in coco_id_to_yolo_id:
                continue

            yolo_cls = coco_id_to_yolo_id[coco_cat_id]
            bbox = ann["bbox"]
            xc, yc, w, h = coco_bbox_to_yolo(bbox, img_w, img_h)

            # sanity clamp
            xc = min(max(xc, 0.0), 1.0)
            yc = min(max(yc, 0.0), 1.0)
            w  = min(max(w,  0.0), 1.0)
            h  = min(max(h,  0.0), 1.0)

            label_lines.append(f"{yolo_cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        label_path = out_label_dir / (Path(file_name).stem + ".txt")
        if len(label_lines) == 0:
            n_no_ann += 1
            label_path.touch()
        else:
            with open(label_path, "w") as f:
                f.write("\n".join(label_lines))

    print(f"Done! Images with NO annotations: {n_no_ann}")

In [13]:
### Run conversion for train & test -> YOLO train/val ###

convert_coco_split_to_yolo(
    coco_json_path=RUOD_ANN_TRAIN,
    img_src_dir=RUOD_IMG_TRAIN,
    out_img_dir=YOLO_IMG_TR,
    out_label_dir=YOLO_LAB_TR,
    coco_id_to_yolo_id=coco_id_to_yolo_id,
)

convert_coco_split_to_yolo(
    coco_json_path=RUOD_ANN_TEST,
    img_src_dir=RUOD_IMG_TEST,
    out_img_dir=YOLO_IMG_VAL,
    out_label_dir=YOLO_LAB_VAL,
    coco_id_to_yolo_id=coco_id_to_yolo_id,
)

Converting 9800 images from /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_pic/train


100%|██████████| 9800/9800 [00:01<00:00, 7028.44it/s]


Done! Images with NO annotations: 0
Converting 4200 images from /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/RUOD/RUOD_pic/test


100%|██████████| 4200/4200 [00:00<00:00, 7544.41it/s]

Done! Images with NO annotations: 0


In [14]:
### some sanity checks ###

n_train_imgs = len(list(YOLO_IMG_TR.glob("*.jpg"))) + len(list(YOLO_IMG_TR.glob("*.png")))
n_train_labels = len(list(YOLO_LAB_TR.glob("*.txt")))
n_val_imgs = len(list(YOLO_IMG_VAL.glob("*.jpg"))) + len(list(YOLO_IMG_VAL.glob("*.png")))
n_val_labels = len(list(YOLO_LAB_VAL.glob("*.txt")))

print("Train images:", n_train_imgs)
print("Train labels:", n_train_labels)
print("Val images:", n_val_imgs)
print("Val labels:", n_val_labels)

Train images: 9800
Train labels: 9800
Val images: 4200
Val labels: 4200


In [15]:
### Write ruod.yaml ###

yaml_dict = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    # we can also add test later if needed
    "names": yolo_id_to_name,  # id -> name
}

yaml_path = ROOT / "ruod.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

print("Wrote YAML to:", yaml_path)
print("\nYAML contents:\n")
print(yaml.dump(yaml_dict))

Wrote YAML to: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/ruod.yaml

YAML contents:

names:
  0: holothurian
  1: echinus
  2: scallop
  3: starfish
  4: fish
  5: corals
  6: diver
  7: cuttlefish
  8: turtle
  9: jellyfish
path: /Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/data/ruod_yolo
train: images/train
val: images/val



In [ ]:
# Jupyter cell 10: train YOLOv8 on RUOD (this can take a while)
project_dir = "../training logs" # change this
model = YOLO("yolov8s.pt")  # small model to start

results = model.train(
    data=str(yaml_path),
    imgsz=640,
    epochs=100,          # you can increase later
    batch=16,           # adjust based on GPU memory
    workers=4,
    project=project_dir,
    name="yolov8s_ruod",
)
print(f"Best model saved at: {project_dir}/yolov8s_ruod/weights/best.pt")

results

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8s.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>


######################################################################## 100.0%


New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.232 🚀 Python-3.13.3 torch-2.9.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/mistycai/Desktop/fall25/ece253/project/underwater-enhancement/ruod.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8

KeyboardInterrupt: 

In [ ]:
### import CLAHE pipeline ###

import sys
sys.path.append(str(ROOT))

from src.contrast.pipeline import CLAHEContrastEnhancer

# enhancer with the tuned params
enhancer = CLAHEContrastEnhancer(
    tile_size=8,
    clip_limit=2.0,
    nbins=256,
    sigma_thresh=0.20,
    grad_thresh=0.25,
)

In [ ]:
### Functions to run YOLO on raw vs CLAHE and display ###

def show_image_bgr(bgr, title=""):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(rgb)
    plt.axis("off")
    plt.title(title)


def run_yolo_with_and_without_clahe(model, img_path):
    img_path = Path(img_path)
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        print("Could not read", img_path)
        return

    ### raw prediction
    results_raw = model.predict(
        source=bgr,
        imgsz=640,
        verbose=False,
    )
    vis_raw = results_raw[0].plot()  # BGR with boxes

    ### CLAHE-enhanced
    bgr_enh, info = enhancer.enhance_bgr(bgr)
    print("CLAHE applied?", info["applied_clahe"],
          "| σ_L =", info["sigma_L"], "| AG(L) =", info["grad_L"])

    results_enh = model.predict(
        source=bgr_enh,
        imgsz=640,
        verbose=False,
    )
    vis_enh = results_enh[0].plot()

    ### visualize side by side
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    show_image_bgr(vis_raw, title="Raw YOLO prediction")

    plt.subplot(1, 2, 2)
    show_image_bgr(vis_enh, title="CLAHE + YOLO prediction")
    plt.tight_layout()
    plt.show()

In [ ]:
### Test on a few images ###

# here we pick 3 random val images from RUOD
val_imgs = list(YOLO_IMG_VAL.glob("*.jpg"))
print("Total val images:", len(val_imgs))

sample_imgs = random.sample(val_imgs, 3)
for p in sample_imgs:
    print("\nTesting:", p.name)
    run_yolo_with_and_without_clahe(model_ruod, p)

In [ ]:
### Test on our own frames (scuba footage images) ###

my_imgs = [
    ROOT / "data" / "raw" / "3 fish.jpg",
    ROOT / "data" / "raw" / "big fish clear.jpg",
    ROOT / "data" / "raw" / "big fish eating.jpg",
]

for p in my_imgs:
    if p.exists():
        print("\n=== My image:", p.name, "===")
        run_yolo_with_and_without_clahe(model_ruod, p)
    else:
        print("Skipping, not found:", p)

In [ ]:
### Small quantitative comparison ###

from collections import defaultdict

def evaluate_subset_with_and_without_clahe(model, img_paths, conf=0.25):
    stats = {"raw": [], "clahe": []}

    for p in tqdm(img_paths):
        bgr = cv2.imread(str(p))
        if bgr is None:
            continue

        # raw
        r_raw = model.predict(bgr, imgsz=640, verbose=False, conf=conf)[0]
        n_raw = len(r_raw.boxes)
        stats["raw"].append(n_raw)

        # clahe
        bgr_enh, info = enhancer.enhance_bgr(bgr)
        r_enh = model.predict(bgr_enh, imgsz=640, verbose=False, conf=conf)[0]
        n_enh = len(r_enh.boxes)
        stats["clahe"].append(n_enh)

    return stats

# example: use 100 random val images
subset = random.sample(val_imgs, min(100, len(val_imgs)))
stats = evaluate_subset_with_and_without_clahe(model_ruod, subset)

print("Avg #detections / image (raw):  ", np.mean(stats['raw']))
print("Avg #detections / image (CLAHE):", np.mean(stats['clahe']))

In [ ]:
### Validation / test ###

trained_model_path = Path("runs/ruod/yolov8n_ruod/weights/best.pt")
assert trained_model_path.exists(), "best.pt not found; check training output."

model_ruod = YOLO(str(trained_model_path))

metrics = model_ruod.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
)
metrics